In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import cv2


# ===================== 需要你修改的路径 =====================

# 各方法生成图像后提取出来的 DC-HED-Contour 草图文件夹
GENERATED_SKETCH_DIRS = {
    "PaletteGAN": Path(r"C:\Users\35278\Desktop\fashion_coloring_file\pycharm_file\CGAN_Sketch_coloring\pix2pix_code04\generated_results_dc_hed_contour_clean_\sketch_masked"),
    # 如果之后还要算其他模型，继续往下加
    "FlexIcon-adapted": Path(r"C:\Users\35278\Desktop\fashion_coloring_file\pycharm_file\CGAN_Sketch_coloring\FlexIcon\test_output\batch_fake\sketch_masked"),
    "IconFlow-adapted": Path(r"C:\Users\35278\Desktop\fashion_coloring_file\pycharm_file\CGAN_Sketch_coloring\IconFlow\test_results\generated\sketch_masked"),
    "Comicolorization-adapted": Path(r"C:\Users\35278\Desktop\fashion_coloring_file\pycharm_file\CGAN_Sketch_coloring\Comicolorization\output\sketch_masked"),
}

# 原始输入 DC-HED-Contour 草图文件夹
REFERENCE_SKETCH_DIR = Path(r"C:\Users\35278\Desktop\fashion_coloring_file\pycharm_file\CGAN_Sketch_coloring\dataset\line_drawing_dc_hed_contour")

# 服装主体 mask 文件夹
MASK_DIR = Path(r"C:\Users\35278\Desktop\fashion_coloring_file\pycharm_file\CGAN_Sketch_coloring\dataset\mask_dc_hed_contour")

# 输出结果文件夹
OUTPUT_DIR = Path(r"C:\Users\35278\Desktop\fashion_coloring_file\pycharm_file\CGAN_Sketch_coloring\evaluation_methods\structure_metric_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ===================== 参数设置 =====================

# mask 中像素值 > 127 的区域认为是服装主体
MASK_THRESHOLD = 127

# 草图线条模式：
# "auto"：自动判断黑线白底还是白线黑底
# "dark"：黑色线条是边缘
# "light"：白色线条是边缘
SKETCH_LINE_MODE = "auto"

# 草图二值化阈值
SKETCH_THRESHOLD = 127

# 边缘匹配容差，单位为像素
# 2 表示允许 2 像素以内的轻微偏移
MATCH_TOLERANCE = 2

# 支持的图片格式
IMAGE_EXTS = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]

In [2]:
def find_images(folder):
    """
    查找文件夹中的图片，支持大小写后缀。
    """
    folder = Path(folder)
    image_paths = []

    for ext in IMAGE_EXTS:
        image_paths.extend(folder.glob(f"*{ext}"))
        image_paths.extend(folder.glob(f"*{ext.upper()}"))

    return sorted(image_paths)


def find_matching_file(folder, stem):
    """
    根据 stem 在 folder 中查找同名图片。
    例如 stem='0001'，可以匹配 0001.png / 0001.jpg。
    """
    folder = Path(folder)

    for ext in IMAGE_EXTS:
        path = folder / f"{stem}{ext}"
        if path.exists():
            return path

        path_upper = folder / f"{stem}{ext.upper()}"
        if path_upper.exists():
            return path_upper

    return None


def normalize_stem(stem):
    """
    如果生成草图文件名带后缀，可以在这里统一去掉。
    例如：
        0001_fake -> 0001
        0001_generated -> 0001
        0001_output -> 0001

    如果你的文件名本来完全一致，这个函数不用改。
    """
    suffixes_to_remove = [
        "_fake",
        "_generated",
        "_output",
        "_result",
        "_sketch",
    ]

    for suffix in suffixes_to_remove:
        if stem.endswith(suffix):
            stem = stem[:-len(suffix)]

    return stem

In [3]:
def load_mask(mask_path, target_size=None, threshold=127):
    """
    读取服装 mask。
    假设：
        服装区域 = 白色 / 255
        背景区域 = 黑色 / 0
    """
    mask_img = Image.open(mask_path).convert("L")

    if target_size is not None and mask_img.size != target_size:
        mask_img = mask_img.resize(target_size, Image.NEAREST)

    mask_arr = np.asarray(mask_img)
    mask = mask_arr > threshold

    return mask


def load_sketch_as_edges(sketch_path, target_size=None, mode="auto", threshold=127):
    """
    将 DC-HED-Contour 草图读取为二值边缘图。

    mode:
        "dark"  : 黑色线条为边缘，白色背景
        "light" : 白色线条为边缘，黑色背景
        "auto"  : 自动判断
    """
    sketch_img = Image.open(sketch_path).convert("L")

    if target_size is not None and sketch_img.size != target_size:
        sketch_img = sketch_img.resize(target_size, Image.NEAREST)

    arr = np.asarray(sketch_img)

    if mode == "dark":
        edges = arr < threshold

    elif mode == "light":
        edges = arr > threshold

    elif mode == "auto":
        # 平均亮度较高，一般是白底黑线
        if arr.mean() > 127:
            edges = arr < threshold
        # 平均亮度较低，一般是黑底白线
        else:
            edges = arr > threshold

    else:
        raise ValueError("mode should be 'auto', 'dark', or 'light'.")

    return edges

In [4]:
def make_ellipse_kernel(radius):
    """
    构造椭圆膨胀核，用于边缘匹配容差。
    radius=2 表示允许 2 像素以内偏移。
    """
    if radius <= 0:
        return np.ones((1, 1), dtype=np.uint8)

    size = radius * 2 + 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))
    return kernel


def dilate_binary(edge_map, radius):
    """
    对二值边缘图进行膨胀。
    """
    kernel = make_ellipse_kernel(radius)
    edge_uint8 = edge_map.astype(np.uint8) * 255
    dilated = cv2.dilate(edge_uint8, kernel, iterations=1)
    return dilated > 0


def compute_edge_f1(pred_edges, ref_edges, mask=None, tolerance=2):
    """
    计算 DC-HED-Contour-based Edge Precision / Recall / F1。

    pred_edges:
        生成草图边缘，来自生成图像的 DC-HED-Contour 草图

    ref_edges:
        输入草图边缘，来自原始输入 DC-HED-Contour 草图

    mask:
        服装主体区域，只用于限定计算区域

    tolerance:
        边缘匹配容差
    """
    pred_edges = pred_edges.astype(bool)
    ref_edges = ref_edges.astype(bool)

    if mask is not None:
        mask = mask.astype(bool)
        pred_edges = pred_edges & mask
        ref_edges = ref_edges & mask

    pred_count = pred_edges.sum()
    ref_count = ref_edges.sum()

    if pred_count == 0 and ref_count == 0:
        return 1.0, 1.0, 1.0

    if pred_count == 0 or ref_count == 0:
        return 0.0, 0.0, 0.0

    # 对参考边缘膨胀，用于计算 Precision
    ref_dilated = dilate_binary(ref_edges, tolerance)

    # 对生成边缘膨胀，用于计算 Recall
    pred_dilated = dilate_binary(pred_edges, tolerance)

    # Precision：生成边缘中，有多少能匹配到参考草图边缘
    matched_pred = pred_edges & ref_dilated
    precision = matched_pred.sum() / pred_count

    # Recall：参考草图边缘中，有多少被生成草图保留下来
    matched_ref = ref_edges & pred_dilated
    recall = matched_ref.sum() / ref_count

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return float(precision), float(recall), float(f1)

In [5]:
def compute_structure_metric_for_one_sample(
    generated_sketch_path,
    reference_sketch_path,
    mask_path,
    mask_threshold=127,
    sketch_line_mode="auto",
    sketch_threshold=127,
    tolerance=2
):
    """
    对单个样本计算 DC-HED-Contour-based Edge F1。
    """

    # 用生成草图尺寸作为目标尺寸
    generated_img = Image.open(generated_sketch_path).convert("L")
    target_size = generated_img.size

    pred_edges = load_sketch_as_edges(
        generated_sketch_path,
        target_size=target_size,
        mode=sketch_line_mode,
        threshold=sketch_threshold
    )

    ref_edges = load_sketch_as_edges(
        reference_sketch_path,
        target_size=target_size,
        mode=sketch_line_mode,
        threshold=sketch_threshold
    )

    mask = load_mask(
        mask_path,
        target_size=target_size,
        threshold=mask_threshold
    )

    precision, recall, f1 = compute_edge_f1(
        pred_edges=pred_edges,
        ref_edges=ref_edges,
        mask=mask,
        tolerance=tolerance
    )

    result = {
        "edge_precision": precision,
        "edge_recall": recall,
        "edge_f1": f1,
        "generated_edge_pixels": int((pred_edges & mask).sum()),
        "reference_edge_pixels": int((ref_edges & mask).sum()),
        "garment_pixels": int(mask.sum()),
    }

    return result

In [6]:
all_rows = []
missing_files = []

for method_name, generated_sketch_dir in GENERATED_SKETCH_DIRS.items():
    generated_sketch_dir = Path(generated_sketch_dir)

    if not generated_sketch_dir.exists():
        raise FileNotFoundError(f"Generated sketch folder not found: {generated_sketch_dir}")

    generated_sketch_paths = find_images(generated_sketch_dir)

    if len(generated_sketch_paths) == 0:
        raise FileNotFoundError(f"No generated sketches found in folder: {generated_sketch_dir}")

    print(f"\nProcessing method: {method_name}")
    print(f"Number of generated sketches: {len(generated_sketch_paths)}")

    for generated_sketch_path in generated_sketch_paths:
        original_stem = generated_sketch_path.stem
        stem = normalize_stem(original_stem)

        reference_sketch_path = find_matching_file(REFERENCE_SKETCH_DIR, stem)
        mask_path = find_matching_file(MASK_DIR, stem)

        if reference_sketch_path is None:
            missing_files.append({
                "method": method_name,
                "generated_sketch": generated_sketch_path.name,
                "missing_type": "reference_sketch",
                "expected_stem": stem
            })
            continue

        if mask_path is None:
            missing_files.append({
                "method": method_name,
                "generated_sketch": generated_sketch_path.name,
                "missing_type": "mask",
                "expected_stem": stem
            })
            continue

        try:
            metrics = compute_structure_metric_for_one_sample(
                generated_sketch_path=generated_sketch_path,
                reference_sketch_path=reference_sketch_path,
                mask_path=mask_path,
                mask_threshold=MASK_THRESHOLD,
                sketch_line_mode=SKETCH_LINE_MODE,
                sketch_threshold=SKETCH_THRESHOLD,
                tolerance=MATCH_TOLERANCE
            )

            row = {
                "method": method_name,
                "generated_sketch": generated_sketch_path.name,
                "reference_sketch": reference_sketch_path.name,
                "mask": mask_path.name,
                "matched_stem": stem,
            }

            row.update(metrics)
            all_rows.append(row)

        except Exception as e:
            print(f"Error processing {generated_sketch_path.name}: {e}")

results_df = pd.DataFrame(all_rows)

if len(results_df) == 0:
    raise RuntimeError("No valid generated-sketch/reference-sketch/mask triplets were processed.")

results_df.head()


Processing method: PaletteGAN
Number of generated sketches: 20152

Processing method: FlexIcon-adapted
Number of generated sketches: 20152

Processing method: IconFlow-adapted
Number of generated sketches: 20152

Processing method: Comicolorization-adapted
Number of generated sketches: 20152


,method,generated_sketch,reference_sketch,mask,matched_stem,edge_precision,edge_recall,edge_f1,generated_edge_pixels,reference_edge_pixels,garment_pixels
0,PaletteGAN,000001.jpg,000001.jpg,000001.png,000001,1.000000,0.915464,0.955867,5218,4921,22556
1,PaletteGAN,000001.jpg,000001.jpg,000001.png,000001,1.000000,0.915464,0.955867,5218,4921,22556
2,PaletteGAN,000002.jpg,000002.jpg,000002.png,000002,0.999813,0.821876,0.902154,5348,5906,22507
3,PaletteGAN,000002.jpg,000002.jpg,000002.png,000002,0.999813,0.821876,0.902154,5348,5906,22507
4,PaletteGAN,000003.jpg,000003.jpg,000003.png,000003,0.997692,0.772714,0.870908,5633,6736,22489


In [7]:
summary_df = (
    results_df
    .groupby("method")
    .agg(
        num_images=("generated_sketch", "count"),

        edge_precision_mean=("edge_precision", "mean"),
        edge_precision_std=("edge_precision", "std"),

        edge_recall_mean=("edge_recall", "mean"),
        edge_recall_std=("edge_recall", "std"),

        edge_f1_mean=("edge_f1", "mean"),
        edge_f1_std=("edge_f1", "std"),

        generated_edge_pixels_mean=("generated_edge_pixels", "mean"),
        reference_edge_pixels_mean=("reference_edge_pixels", "mean"),
        garment_pixels_mean=("garment_pixels", "mean"),
    )
    .reset_index()
)

summary_df

,method,num_images,edge_precision_mean,edge_precision_std,edge_recall_mean,edge_recall_std,edge_f1_mean,edge_f1_std,generated_edge_pixels_mean,reference_edge_pixels_mean,garment_pixels_mean
0,Comicolorization-adapted,20152,0.971848,0.044131,0.830188,0.123436,0.888942,0.075510,7592.230746,8604.415840,35928.830985
1,FlexIcon-adapted,20152,0.976555,0.034665,0.942672,0.056057,0.957990,0.034424,7049.677352,6516.902938,27197.988884
2,IconFlow-adapted,20152,0.962427,0.044486,0.914814,0.078704,0.935189,0.046537,8861.884478,8604.415840,35928.830985
3,PaletteGAN,20152,0.984071,0.018758,0.942800,0.056654,0.961893,0.031564,6697.418817,6265.693628,26162.930726


In [8]:
per_image_csv = OUTPUT_DIR / "dc_hed_contour_edge_f1_per_image.csv"
summary_csv = OUTPUT_DIR / "dc_hed_contour_edge_f1_summary.csv"

results_df.to_csv(per_image_csv, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

print(f"Per-image results saved to: {per_image_csv}")
print(f"Summary results saved to: {summary_csv}")

if len(missing_files) > 0:
    missing_df = pd.DataFrame(missing_files)
    missing_csv = OUTPUT_DIR / "missing_dc_hed_contour_edge_f1_files.csv"
    missing_df.to_csv(missing_csv, index=False, encoding="utf-8-sig")
    print(f"Missing file list saved to: {missing_csv}")
else:
    print("No missing files.")

Per-image results saved to: C:\Users\35278\Desktop\fashion_coloring_file\pycharm_file\CGAN_Sketch_coloring\evaluation_methods\structure_metric_results\dc_hed_contour_edge_f1_per_image.csv
Summary results saved to: C:\Users\35278\Desktop\fashion_coloring_file\pycharm_file\CGAN_Sketch_coloring\evaluation_methods\structure_metric_results\dc_hed_contour_edge_f1_summary.csv
No missing files.
